<font size="6" color='grey'><b>KI-Agenten. Verstehen. Anwenden. Gestalten.</b></font> </br>

<font size="5" color='grey'><b>MCP + HuggingFace — KI-Modelle als MCP-Tools</b></font> </br>

---

In [3]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q fastmcp langchain-mcp-adapters huggingface_hub

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M34-MCP-HuggingFace"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, mermaid, show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt
✓ HF_TOKEN erfolgreich gesetzt

Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.10
langchain-chroma                         1.1.0
langchain-classic                        1.0.2
langchain-community                      0.4.1
langchain-core                           1.2.18
langchain-mcp-adapters                   0.2.1
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.0.10
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.9

IP-Adresse: 136.117.116.28
Hostname: 28.116.117.136.bc.googleusercontent.com
Stadt: The Dalles
Region: Oregon
Land: US
Koordinaten: 45.5946,-121.1787
Provi

# 1 | Übersicht
---

## Idee: Crypto-Tools als öffentlicher MCP-Server auf HF Spaces

Statt einen lokalen MCP-Server zu betreiben, läuft der FastMCP-Server **permanent auf Hugging Face Spaces** — öffentlich erreichbar, kein lokaler Setup notwendig.

**Das Problem:** Tools wie Caesar-Cipher oder Vigenère sind nützliche Algorithmen, aber ein Agent müsste deren Implementierung selbst kennen.

**Die Lösung:** Ein **FastMCP-Server auf HF Spaces** wrапpt die Algorithmen. Der Agent sieht einfache, klar benannte Tools — ohne Implementierungsdetails.

| Schicht | Was passiert |
|---------|-------------|
| **Agent (gpt-4o-mini)** | Entscheidet, welches Tool gebraucht wird |
| **MCP Client** | Leitet Tool-Calls zum FastMCP-Server weiter |
| **FastMCP Server (HF Space)** | Stellt Crypto-Tools bereit — öffentlich erreichbar |
| **Tool-Logik** | Reine Python-Implementierungen der Algorithmen |

**Vorteile dieses Musters:**
- ✅ Kein lokaler Server notwendig — HF Space läuft permanent
- ✅ Öffentlich erreichbar von überall (kein API-Key für die Tools nötig)
- ✅ Agent muss keine Algorithmus-Details kennen — nur die Tool-Namen
- ✅ Gleicher `MultiServerMCPClient`-Code wie bei lokalen Servern

In [4]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: MCP + HF Space Architektur</font> </br></p>

diagram = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    U(["👤 User"])
    A["LangGraph Agent\ngpt-4o-mini"]
    C["MCP Client\nMultiServerMCPClient"]
    S["FastMCP Server\nHF Space simple_mcp\nhttps://ralf42-simple_mcp.hf.space/mcp"]

    T1["🔑 caesar_encrypt\ncaesar_decrypt"]
    T2["🔄 rot13"]
    T3["🗝️ vigenere_encrypt\nvigenere_decrypt"]
    T4["💾 text_to_binary\nbinary_to_text"]

    U -->|Anfrage| A
    A -->|Tool Call| C
    C -->|HTTP POST JSON-RPC| S
    S --> T1 & T2 & T3 & T4
    T1 & T2 & T3 & T4 -->|Ergebnis| S
    S -->|Tool Result| C
    C -->|ToolMessage| A
    A -->|Antwort| U

    style A  fill:#2E7D32,color:#fff
    style C  fill:#1565C0,color:#fff
    style S  fill:#37474F,color:#fff
    style U  fill:#E65100,color:#fff
    style T1 fill:#4A148C,color:#fff
    style T2 fill:#4A148C,color:#fff
    style T3 fill:#4A148C,color:#fff
    style T4 fill:#4A148C,color:#fff
'''
mermaid(diagram, width=800)

# 2 | HF-MCP-Server erstellen
---

Der FastMCP-Server auf HF Spaces stellt 7 Crypto-Tools bereit:

| Tool | Algorithmus | Aufgabe |
|------|-------------|---------|
| `caesar_encrypt` | Caesar-Cipher | Text mit Shift (1–25) verschlüsseln |
| `caesar_decrypt` | Caesar-Cipher | Caesar-Text entschlüsseln |
| `rot13` | ROT13 | ROT13-Kodierung / Dekodierung (ist eigene Umkehrfunktion) |
| `vigenere_encrypt` | Vigenère-Cipher | Text mit Schlüsselwort verschlüsseln |
| `vigenere_decrypt` | Vigenère-Cipher | Vigenère-Text entschlüsseln |
| `text_to_binary` | Binär | Text in 8-Bit Binärdarstellung konvertieren |
| `binary_to_text` | Binär | Binärdarstellung zurück in Text dekodieren |

> ℹ️ **HF Space URL:** `https://ralf42-simple-mcp.hf.space/mcp`
> Kein lokaler Server, kein HF-API-Key für die Crypto-Tools erforderlich.
> HF Spaces schlafen nach Inaktivität ein — der erste Request kann 10–30 s dauern.

In [ ]:
#@title 🔌 HF Space aufwecken + Verbindung prüfen{ display-mode: "form" }
import requests, time, os

SPACE_ID   = "Ralf42/simple_mcp"
SPACE_URL  = "https://ralf42-simple-mcp.hf.space"
MCP_URL    = f"{SPACE_URL}/mcp"
API_URL    = f"https://huggingface.co/api/spaces/{SPACE_ID}"
HF_TOKEN   = os.getenv("HF_TOKEN", "").strip()        # .strip() entfernt \r\n
HEADERS    = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}

mprint(f"### 🔌 HF Space MCP-Server\n---")
print(f"Space:    {SPACE_ID}")
print(f"URL:      {SPACE_URL}")
print(f"Endpoint: {MCP_URL}")
print()

# ── Schritt 1: Aktuellen Status prüfen ───────────────────────────────────────
r = requests.get(API_URL, headers=HEADERS, timeout=10)
if r.ok:
    stage = r.json().get("runtime", {}).get("stage", "UNKNOWN")
    print(f"Status: {stage}")
    if "ERROR" in stage:
        print("❌ Space im Fehlerstatus — bitte app.py im HF Space korrigieren und neu deployen")
        raise SystemExit
    elif stage == "RUNNING":
        print("✅ Space bereits aktiv")
    else:
        print("⏳ Wecke Space auf ...")
        try:
            requests.get(SPACE_URL, timeout=10)
        except Exception:
            pass

        for i in range(24):   # max. 120 s
            time.sleep(5)
            r2 = requests.get(API_URL, headers=HEADERS, timeout=5)
            if r2.ok:
                stage = r2.json().get("runtime", {}).get("stage", "?")
                print(f"  [{(i+1)*5:>3}s] {stage}")
                if stage == "RUNNING":
                    break
                if "ERROR" in stage:
                    print("❌ Space im Fehlerstatus — bitte Logs prüfen (Zelle unten)")
                    raise SystemExit
        else:
            print("⚠️ Timeout — Zelle erneut ausführen")

# ── Schritt 2: MCP-Endpoint testen ───────────────────────────────────────────
print()
try:
    r = requests.post(
        MCP_URL,
        json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
        headers={"Content-Type": "application/json"},
        timeout=15
    )
    if r.status_code in [200, 400, 406]:
        print(f"✅ MCP-Endpoint erreichbar — HTTP {r.status_code}")
    elif r.status_code == 404:
        print(f"❌ HTTP 404 — /mcp-Route nicht gefunden")
        print(f"   → app.py prüfen: app.mount('/mcp', mcp.http_app())")
    else:
        print(f"⚠️ HTTP {r.status_code}")
except Exception as e:
    print(f"❌ {e}")

In [ ]:
#@title 🩺 HF Space Status + Logs prüfen{ display-mode: "form" }
import requests, os

HF_TOKEN   = os.getenv("HF_TOKEN", "").strip()        # .strip() entfernt \r\n
SPACE_ID   = "Ralf42/simple_mcp"
HEADERS    = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
BASE_URL   = f"https://huggingface.co/api/spaces/{SPACE_ID}"

# ── Space-Status ──────────────────────────────────────────────────────────────
r = requests.get(BASE_URL, headers=HEADERS, timeout=10)
if r.ok:
    info    = r.json()
    stage   = info.get("runtime", {}).get("stage", "UNKNOWN")
    sdk     = info.get("sdk", "?")
    print(f"📦 Space:  {SPACE_ID}")
    print(f"🔧 SDK:    {sdk}")
    print(f"🟢 Status: {stage}")
    if stage == "SLEEPING":
        print("   ⚠️  Space schläft — erster Request weckt ihn auf (~30 s)")
    elif stage == "RUNNING":
        print(f"   MCP-Endpoint: https://ralf42-simple-mcp.hf.space/mcp")
else:
    print(f"⚠️ Status-API: HTTP {r.status_code}")

# ── Run-Logs (letzte Zeilen) ──────────────────────────────────────────────────
print("\n--- Run-Logs (letzte Zeilen) ---")
try:
    with requests.get(f"{BASE_URL}/logs/run", headers=HEADERS, stream=True, timeout=5) as log_r:
        lines = []
        for raw in log_r.iter_lines():
            if raw:
                line = raw.decode("utf-8", errors="replace")
                if line.startswith("data:"):
                    import json as _json
                    try:
                        entry = _json.loads(line[5:].strip())
                        msg = entry.get("data", {}).get("message", "")
                        if msg:
                            lines.append(msg)
                    except Exception:
                        pass
            if len(lines) >= 15:
                break
        for l in lines[-10:]:
            print(f"  {l}")
except Exception as e:
    print(f"  (Logs: {e})")

print(f"\n📎 Build-Logs: curl -N -H 'Authorization: Bearer $HF_TOKEN' '{BASE_URL}/logs/build'")
print(f"📎 Run-Logs:   curl -N -H 'Authorization: Bearer $HF_TOKEN' '{BASE_URL}/logs/run'")

In [ ]:
#@title 📋 Verfügbare Tools anzeigen{ display-mode: "form" }
import requests, json

r = requests.post(
    MCP_URL,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
    headers={"Content-Type": "application/json"},
    timeout=15
)

if r.status_code in [200, 400, 406]:
    try:
        tools = r.json().get("result", {}).get("tools", [])
        print(f"📋 {len(tools)} Tools registriert:\n")
        for t in tools:
            print(f"• {t['name']}")
            print(f"  {t.get('description', '')[:100]}")
            print()
    except Exception:
        print(f"Server antwortet (HTTP {r.status_code}) — Tool-Liste nicht parsebar")
else:
    print(f"⚠️ Status: {r.status_code}")

# 3 | LangGraph Agent mit HF-Tools
---

Der `MultiServerMCPClient` lädt die Tool-Definitionen vom HF Space. Der LangGraph ReAct-Agent wählt automatisch das passende Crypto-Tool — basierend auf den Docstrings der Tools.

**Wichtig:** HF Spaces schlafen nach Inaktivität ein (Free Tier). Beim ersten Request kann es 10–30 Sekunden dauern, bis der Space aufgewacht ist. Folge-Requests sind sofort schnell.

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  sequenceDiagram: Handshake + Tool-Aufruf</font> </br></p>

diagram = '''
sequenceDiagram
    autonumber
    actor User
    participant Agent as LangGraph Agent
    participant Client as MCP Client
    participant Server as FastMCP Server (HF Space)

    Note over Agent,Server: Phase 1 — Tool Discovery
    Client->>Server: HTTP POST (initialize)
    Server-->>Client: Capabilities
    Client->>Server: HTTP POST (tools/list)
    Server-->>Client: [caesar_encrypt, caesar_decrypt, rot13, ...]

    Note over Agent,Server: Phase 2 — Anfrage verarbeiten
    User->>Agent: "Verschlüssele HALLO mit Caesar Shift 3"
    Agent->>Client: Tool: caesar_encrypt(text="HALLO", shift=3)
    Client->>Server: HTTP POST JSON-RPC
    Server-->>Client: "KDOOR"
    Client-->>Agent: ToolMessage
    Agent->>User: "Das verschlüsselte Wort lautet: KDOOR"
'''
mermaid(diagram, width=950)

In [ ]:
#@title 🔌 MCP Client + Tools laden{ display-mode: "form" }
from langchain_mcp_adapters.client import MultiServerMCPClient

client_crypto = MultiServerMCPClient({
    "crypto": {
        "transport": "streamable_http",
        "url": "https://ralf42-simple-mcp.hf.space/mcp"
    }
})

crypto_tools = await client_crypto.get_tools()

print(f"✅ {len(crypto_tools)} Crypto-Tools geladen:")
for t in crypto_tools:
    print(f"   • {t.name}")
    print(f"     {t.description[:80]}")

In [ ]:
#@title 🤖 LangGraph ReAct-Agent mit Crypto-Tools{ display-mode: "form" }
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from IPython.display import Image as IPImage, display

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

crypto_agent = create_react_agent(
    llm,
    tools=crypto_tools,
    prompt=(
        "Du bist ein Experte für klassische Verschlüsselungsverfahren mit Zugriff auf Crypto-Tools über MCP.\n"
        "Nutze die Tools für alle Verschlüsselungs- und Entschlüsselungsaufgaben.\n"
        "Verfügbare Algorithmen: Caesar-Cipher, ROT13, Vigenère-Cipher, Binärdarstellung.\n"
        "Antworte immer auf Deutsch und erkläre kurz, welchen Algorithmus du verwendet hast."
    ),
)

try:
    display(IPImage(crypto_agent.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print(f"(Graph-Visualisierung: {e})")

print("✅ LangGraph ReAct-Agent mit Crypto-MCP-Tools bereit")

In [ ]:
#@title 🔑 Demo 1: Caesar-Verschlüsselung{ display-mode: "form" }
from langchain_core.messages import HumanMessage

mprint("### 🔑 Caesar-Cipher\n---")

anfragen = [
    "Verschlüssele 'Hallo Welt' mit Caesar Shift 13.",
    "Entschlüssele 'KHOOR' — es wurde mit Caesar Shift 3 verschlüsselt.",
    "Verschlüssele 'KI-Agenten sind mächtig' mit Shift 7 und entschlüssele das Ergebnis wieder.",
]

for anfrage in anfragen:
    result = await crypto_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
    mprint(f"**Anfrage:** {anfrage}\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

In [ ]:
#@title 🔄 Demo 2: ROT13{ display-mode: "form" }
mprint("### 🔄 ROT13\n---")

anfragen_rot13 = [
    "Wende ROT13 auf 'Geheime Nachricht' an.",
    "Dekodiere diesen ROT13-Text: 'Xhafgyvpur Vagryyvatrm'",
    "Warum ist ROT13 seine eigene Umkehrfunktion? Demonstriere es mit dem Wort 'TEST'.",
]

for anfrage in anfragen_rot13:
    result = await crypto_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
    mprint(f"**Anfrage:** {anfrage}\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

In [ ]:
#@title 🗝️ Demo 3: Vigenère-Cipher{ display-mode: "form" }
mprint("### 🗝️ Vigenère-Verschlüsselung\n---")

anfragen_vig = [
    "Verschlüssele 'ANGRIFF UM MITTERNACHT' mit dem Schlüsselwort 'GEHEIM'.",
    "Entschlüssele 'MFVMF' mit dem Schlüsselwort 'KEY'.",
    "Warum ist Vigenère sicherer als Caesar? Demonstriere mit 'HELLO' und Schlüssel 'AB'.",
]

for anfrage in anfragen_vig:
    result = await crypto_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
    mprint(f"**Anfrage:** {anfrage}\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

In [ ]:
#@title 💾 Demo 4: Binärdarstellung{ display-mode: "form" }
mprint("### 💾 Text ↔ Binär\n---")

anfragen_bin = [
    "Konvertiere 'Hallo' in Binärdarstellung.",
    "Dekodiere diesen Binärtext: '01001000 01101001'",
    "Konvertiere 'MCP' in Binär und dann wieder zurück — zeige alle Schritte.",
]

for anfrage in anfragen_bin:
    result = await crypto_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
    mprint(f"**Anfrage:** {anfrage}\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

# 4 | Multi-Task Demo — Tools kombinieren
---

Der Agent kann mehrere Crypto-Tools in einer einzigen Anfrage kombinieren. Zum Beispiel: Text mit Vigenère verschlüsseln und dann in Binär konvertieren, oder mehrere Algorithmen vergleichen.

Das zeigt die Stärke von MCP: Der Agent orchestriert verschiedene Tools wie ein Dirigent — ohne deren Implementierungsdetails zu kennen.

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: Multi-Tool Orchestrierung</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    A(["Anfrage:\nText verschlüsseln\n+ Binär konvertieren"])

    subgraph agent ["LangGraph ReAct Agent"]
        S1["Schritt 1:\nvigenere_encrypt"]
        S2["Schritt 2:\ntext_to_binary"]
    end

    R(["Kombinierte\nAntwort auf Deutsch"])

    A --> S1
    S1 -->|"Verschlüsselter Text"| S2
    S2 --> R

    style agent fill:#1B5E20,color:#fff
    style A    fill:#E65100,color:#fff
    style R    fill:#1565C0,color:#fff
'''
mermaid(diagram, width=700)

In [ ]:
#@title 🔀 Multi-Task: Mehrere Tools kombinieren{ display-mode: "form" }
mprint("### 🔀 Multi-Task Demo\n---")

multi_anfragen = [
    (
        "Führe folgende Schritte durch: "
        "1) Verschlüssele 'HELLO WORLD' mit Vigenère-Schlüssel 'SECRET', "
        "2) Konvertiere das Ergebnis in Binärdarstellung. "
        "Zeige alle Zwischenergebnisse."
    ),
    (
        "Zeige den Unterschied: Verschlüssele 'PYTHON' einmal mit Caesar Shift 3 "
        "und einmal mit ROT13. Vergleiche beide Ergebnisse und erkläre den Unterschied."
    ),
]

for anfrage in multi_anfragen:
    result = await crypto_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
    mprint(f"**Anfrage:** {anfrage[:100]}...\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

In [ ]:
#@title 🛑 Verbindung beenden{ display-mode: "form" }
mprint("### 🛑 MCP-Client schließen\n---")

try:
    await client_crypto.__aexit__(None, None, None)
    print("✅ MCP-Client-Verbindung geschlossen")
except Exception as e:
    print(f"ℹ️ {e}")

print("\nDer HF Space MCP-Server läuft weiter — keine lokalen Prozesse zu beenden.")

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M34-MCP-HuggingFace", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='black' size="5">Eigene Crypto-Tools und MCP-Experimente</font></p>

**Aufgabe 1 — Neues Tool: Atbash-Cipher**
Ergänze den HF Space `simple_mcp` um ein Tool `atbash(text: str) -> str`.
Atbash ersetzt jeden Buchstaben durch seinen gespiegelten Buchstaben (A↔Z, B↔Y, ...).
Teste mit: `"HELLO"` → `"SVOOL"`.

**Aufgabe 2 — Tool: Caesar Brute-Force**
Füge ein Tool `caesar_bruteforce(text: str) -> str` hinzu, das alle 25 möglichen Shifts auflistet.
Teste mit einem unbekannt verschlüsselten Text und lasse den Agenten den richtigen Shift erraten.

**Aufgabe 3 — Zweiter MCP-Server: HF Inference Tools**
Erstelle ein zweites HF Space mit FastMCP, das HF-Inference-API-Tools bereitstellt (z.B. Sentiment-Analyse, Übersetzung).
Kombiniere beide Server in einem `MultiServerMCPClient`:
```python
client = MultiServerMCPClient({
    "crypto": {"transport": "streamable_http", "url": "https://ralf42-simple-mcp.hf.space/mcp"},
    "hf":     {"transport": "streamable_http", "url": "https://ralf42-<dein-space>.hf.space/mcp"},
})
```
Stelle eine Anfrage, die beide Server nutzt (z.B. Text verschlüsseln + Sentiment analysieren).

> 💡 **Tipp:** HF Spaces sind kostenlos. Jedes Space kann ein eigener MCP-Server sein.
> Mit `MultiServerMCPClient` können beliebig viele Server kombiniert werden.

> 💡 **HF Space URL-Schema:** `https://<username>-<space-name>.hf.space/mcp`
> Unterstriche im Space-Namen werden zu Bindestrichen (z.B. `simple_mcp` → `simple-mcp`).